# Fine-tune the reranker on Colab GPU

This trains the **BGE cross-encoder reranker** on your ML-paper data so it judges
*ML-paper* relevance better. This is the one real **model-training** step in the project.

**Steps:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Run every cell top to bottom.
3. When prompted, upload your local `evals/train_pairs.jsonl`.
4. At the end, `bge-reranker-base-ft.zip` downloads. Unzip it into your project at
   `models/bge-reranker-base-ft/`, then re-run the evaluation locally.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), '-',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY (enable GPU!)')

In [ ]:
# Pin the version so the classic CrossEncoder.fit() API is available.
!pip install -q sentence-transformers==3.2.1

In [ ]:
# Upload evals/train_pairs.jsonl when the picker appears.
from google.colab import files
uploaded = files.upload()   # choose train_pairs.jsonl

import json
pairs = [json.loads(l) for l in open('train_pairs.jsonl', encoding='utf-8')]
print(f'Loaded {len(pairs)} query groups.')

In [ ]:
# Build labeled (query, passage) training examples: positive=1, hard negatives=0.
from sentence_transformers import InputExample

examples = []
for p in pairs:
    examples.append(InputExample(texts=[p['query'], p['positive']], label=1.0))
    for neg in p['negatives']:
        examples.append(InputExample(texts=[p['query'], neg], label=0.0))
print(f'{len(examples)} labeled training examples '
      f"({sum(e.label==1.0 for e in examples)} positive, "
      f"{sum(e.label==0.0 for e in examples)} negative)")

In [ ]:
# Fine-tune the cross-encoder (a few minutes on the GPU).
from sentence_transformers.cross_encoder import CrossEncoder
from torch.utils.data import DataLoader

model = CrossEncoder('BAAI/bge-reranker-base', num_labels=1, max_length=512)
loader = DataLoader(examples, shuffle=True, batch_size=16)
model.fit(
    train_dataloader=loader,
    epochs=2,
    warmup_steps=100,
    output_path='bge-reranker-base-ft',
)
print('Done training -> saved to bge-reranker-base-ft/')

In [ ]:
# Zip the fine-tuned model and download it.
import shutil
shutil.make_archive('bge-reranker-base-ft', 'zip', 'bge-reranker-base-ft')
from google.colab import files
files.download('bge-reranker-base-ft.zip')
print('Unzip into your project at models/bge-reranker-base-ft/ , then re-run the eval.')